# Project: Classifying breast tumours

*ML Academy — complete solution*

**Goal.** Build a classifier that flags malignant tumours, choose between five algorithms fairly, and tune it so that it almost never misses a malignant case.

**The data.** 569 breast-tumour biopsies from the Wisconsin Diagnostic Breast Cancer study (Street, Wolberg and Mangasarian, 1993), shipped with scikit-learn as load_breast_cancer. Each has 30 measurements of the cell nuclei in a digitised image (radius, texture, smoothness, …) and a diagnosis: malignant or benign.

**Uses:** Lesson 8, Lesson 9, Lesson 11, Lesson 13, Lesson 14, Lesson 15.

This notebook contains the full solution with outputs. Try the starter notebook first!

Requirements: `pip install numpy pandas scikit-learn matplotlib` — the dataset ships with scikit-learn, so no download is needed.

In [1]:
# Run this cell first: it defines check(), which tells you whether each step is right.
import numpy as np
def check(get, want, hint=''):
    try:
        got = get()
        if isinstance(want, str):
            ok = str(got) == want
        else:
            ok = np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=1e-6)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


## Step 1. Load the data

In scikit-learn's encoding 0 = malignant. We make **malignant the positive class (1)**, because that's the case we must not miss.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
X = data.data
y = (data.target == 0).astype(int)
print(X.shape, "share malignant:", round(y.mean(), 3))

(569, 30) share malignant: 0.373


In [3]:
check(lambda: y.sum(), 212, hint="(data.target == 0).astype(int) turns 'malignant' into 1.")

✅ Correct!


## Step 2. Split, keeping the class balance

Use `stratify=y` so training and test sets have the same share of malignant cases. Keep 25% for testing (`random_state=0`).

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X_train.shape, X_test.shape)

(426, 30) (143, 30)


In [5]:
check(lambda: y_test.sum(), 53, hint='Add stratify=y to train_test_split.')

✅ Correct!


## Step 3. Compare five algorithms with cross-validation

Each model that relies on distances or weights gets a `StandardScaler`. Trees don't need one. We score with 5-fold cross-validated accuracy on the **training set only** — the test set stays untouched.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "kNN (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": DecisionTreeClassifier(max_depth=4, random_state=0),
    "random forest": RandomForestClassifier(n_estimators=300, random_state=0),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC()),
}
cv_acc = {name: cross_val_score(m, X_train, y_train, cv=5).mean() for name, m in models.items()}
best_name = max(cv_acc, key=cv_acc.get)
for name, acc in sorted(cv_acc.items(), key=lambda kv: -kv[1]):
    print(f"{name:20s} {acc:.3f}")
print("best:", best_name)

logistic regression  0.981
SVM (RBF)            0.970
kNN (k=5)            0.967
random forest        0.955
decision tree        0.920
best: logistic regression


In [7]:
check(lambda: best_name, 'logistic regression', hint='A dict comprehension: {name: cross_val_score(m, X_train, y_train, cv=5).mean() for name, m in models.items()}.')
check(lambda: cv_acc['random forest'], 0.955403556771546, hint='Use cv=5 and the default accuracy scoring.')

✅ Correct!
✅ Correct!


## Step 4. Test the winner once

Fit the best model on the full training set and evaluate it on the test set — once. Look at the confusion matrix: which kind of mistake does it make?

In [8]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score
best = models[best_name].fit(X_train, y_train)
pred = best.predict(X_test)
cm = confusion_matrix(y_test, pred)
recall = recall_score(y_test, pred)
precision = precision_score(y_test, pred)
print(cm)
print(f"accuracy {accuracy_score(y_test, pred):.3f} · precision {precision:.3f} · recall {recall:.3f}")

[[90  0]
 [ 2 51]]
accuracy 0.986 · precision 1.000 · recall 0.962


In [9]:
check(lambda: recall, 0.9622641509433962, hint='recall_score(y_test, pred): the share of malignant tumours it caught.')
check(lambda: cm.tolist(), [[90, 0], [2, 51]], hint='confusion_matrix(y_test, pred).')

✅ Correct!
✅ Correct!


## Step 5. Never miss a malignant tumour: lower the threshold

A missed cancer is far worse than a false alarm that leads to another test. Use predicted probabilities and lower the threshold until recall reaches 1.0, then see what it costs in precision (Lesson 9).

In [10]:
proba = best.predict_proba(X_test)[:, 1]
for t in [0.5, 0.3, 0.2, 0.1]:
    p_t = (proba >= t).astype(int)
    print(f"threshold {t}: precision {precision_score(y_test, p_t):.3f}, recall {recall_score(y_test, p_t):.3f}, missed {int(((p_t == 0) & (y_test == 1)).sum())}")
low = (proba >= 0.1).astype(int)
recall_low = recall_score(y_test, low)

threshold 0.5: precision 1.000, recall 0.962, missed 2
threshold 0.3: precision 0.895, recall 0.962, missed 2
threshold 0.2: precision 0.852, recall 0.981, missed 1
threshold 0.1: precision 0.828, recall 1.000, missed 0


In [11]:
check(lambda: recall_low, 1.0, hint='Compare proba with the threshold: (proba >= t).astype(int).')

✅ Correct!


## Step 6. Which measurements matter?

Random forests report how much each feature reduced impurity (Lesson 14).

In [12]:
forest = models["random forest"].fit(X_train, y_train)
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(5).round(3)

mean concave points     0.143
worst concave points    0.118
worst perimeter         0.117
worst radius            0.111
worst area              0.086
dtype: float64

In [13]:
check(lambda: importances.index[0], 'mean concave points', hint='pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False).')

✅ Correct!


## Go further

- Tune C for logistic regression with GridSearchCV.
- Draw the ROC curve with RocCurveDisplay.from_estimator and read off the AUC.
- Which threshold would you choose, and how would you explain the trade-off to a doctor?